<a href="https://colab.research.google.com/github/keswong/phd_listing_repo/blob/main/3_11_4_Comparing_Workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%reset -f
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

df = pd.read_excel(".../coding_teacher_automated_interventions.xlsx", sheet_name="automated-interventions")
df.fillna(0, inplace=True)

print("\n==============================")
print("FULL CONTINGENCY TABLE")
print("==============================")
print(df)

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C3","C4","PRS1","PRS2","PRS3",
    "PRT4","PRT5","PRT6","PRT7","PRT_M1","PRT_M2","PRT_M4","PRT_M5"
]
group_col = "condition"  # 'with_pedagogy' vs 'without_pedagogy'
results = []

for col in coding_cols:
    contingency = pd.crosstab(df[col], df[group_col])
    contingency.index = contingency.index.astype(str)
    contingency.columns = contingency.columns.astype(str)
    print(f"\n--- {col} ---")
    print(contingency)

    # --- Enforce 2x2 structure ---
    if contingency.shape != (2, 2):
        results.append({
            'Variable': col,
            'Total_N': contingency.values.sum(),
            'Odds_Ratio': None,
            'P-value': None,
            'Test': 'Invalid (not 2x2)',
            'With Pedagogy (# coded 1)': None,
            'Without Pedagogy (# coded 1)': None,
            'With Pedagogy (# coded 0)': contingency.iloc[0, 0] if 0 in contingency.index else None,
            'Without Pedagogy (# coded 0)': contingency.iloc[0, 1] if 1 in contingency.columns else None
        })
        continue

    total = contingency.values.sum()

    # Fisher's exact test
    odds_ratio, p_value = fisher_exact(contingency)

    print(f"  Fisher's Exact Test: P-value = {p_value:.4g}, OR = {odds_ratio:.3f}")

    # --- Store results ---
    results.append({
        'Variable': col,
        'Total_N': total,
        'Odds_Ratio': odds_ratio,
        'P-value': p_value,
        'Test': "Fisher's Exact",
        'With Pedagogy (# coded 1)': contingency.iloc[1, 0],
        'Without Pedagogy (# coded 1)': contingency.iloc[1, 1],
        'With Pedagogy (# coded 0)': contingency.iloc[0, 0],
        'Without Pedagogy (# coded 0)': contingency.iloc[0, 1]
    })

# Create results DataFrame
results_df = pd.DataFrame(results)

# ============================================
# Multiple testing corrections using multipletests
# ============================================

# Filter valid tests (2x2 tables)
valid_mask = results_df['Test'] != 'Invalid (not 2x2)'
valid_pvalues = results_df.loc[valid_mask, 'P-value'].values

if len(valid_pvalues) > 0:
    # Holm-Bonferroni correction
    holm_corrected = multipletests(valid_pvalues, alpha=0.05, method='holm')[1]

    # Bonferroni correction (for comparison)
    bonferroni_corrected = multipletests(valid_pvalues, alpha=0.05, method='bonferroni')[1]

    # Add back to DataFrame
    results_df.loc[valid_mask, 'P-value_Holm'] = holm_corrected
    results_df.loc[valid_mask, 'P-value_Bonferroni'] = bonferroni_corrected

    # Significance flags
    results_df.loc[valid_mask, 'Significant (Holm)'] = holm_corrected < 0.05
    results_df.loc[valid_mask, 'Significant (Bonferroni)'] = bonferroni_corrected < 0.05

# ============================================
# Display results
# ============================================

print("\n" + "="*60)
print("FISHER'S EXACT TEST RESULTS WITH MULTIPLE TESTING CORRECTIONS")
print("="*60)
print(f"Number of valid 2x2 tests: {len(valid_pvalues)}")
print(f"Significance level (α): 0.05")
print("\n")

# Show key comparisons
comparison_cols = [
    'Variable', 'Odds_Ratio', 'P-value',
    'P-value_Holm', 'Significant (Holm)',
    'P-value_Bonferroni', 'Significant (Bonferroni)'
]

print(results_df[comparison_cols].to_string(index=False, float_format='%.4f'))

# Show significant findings
print("\n" + "="*60)
print("SIGNIFICANT FINDINGS AFTER CORRECTION")
print("="*60)

holm_sig = results_df[results_df['Significant (Holm)'] == True]
if len(holm_sig) > 0:
    print("\nHolm-Bonferroni significant variables:")
    print(holm_sig[['Variable', 'Odds_Ratio', 'P-value', 'P-value_Holm']].to_string(index=False, float_format='%.4e'))
else:
    print("\nNo variables significant after Holm-Bonferroni correction")

bonf_sig = results_df[results_df['Significant (Bonferroni)'] == True]
if len(bonf_sig) > 0:
    print("\nBonferroni significant variables:")
    print(bonf_sig[['Variable', 'Odds_Ratio', 'P-value', 'P-value_Bonferroni']].to_string(index=False, float_format='%.4e'))


FULL CONTINGENCY TABLE
    utterance_id         condition   M1   M2   M3   M4   M5   M6   M7   F1  \
0            924     with_pedagogy  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
1            924  without_pedagogy  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0   
2            926     with_pedagogy  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
3            926  without_pedagogy  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
4            951     with_pedagogy  1.0  0.0  1.0  0.0  1.0  0.0  0.0  0.0   
..           ...               ...  ...  ...  ...  ...  ...  ...  ...  ...   
57          1094  without_pedagogy  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
58          1096     with_pedagogy  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
59          1096  without_pedagogy  1.0  0.0  1.0  0.0  1.0  0.0  0.0  0.0   
60          1098     with_pedagogy  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
61          1098  without_pedagogy  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   

    ...  PRS2  PRS3  PRT4  PRT5  PRT6  